# ETL Transformación - Carga Inicial

Este notebook procesa la transformación e inserción de datos hacia el DWH. 
Resuelve los problemas anteriores con la tabla de hechos (columnas correctas) y elimina el error CRITICO de foreign keys utilizando DELETE en lugar de TRUNCATE.
Nuevas transformaciones: DNI como varchar preservando puntos, nacionalidad nula -> 'sin especificar', deduplicación de nombres, validación de examen por nota. Cálculo de edad exacto con anio_ingreso.

In [1]:
import logging
import os
import sys
import warnings
from datetime import date
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd().resolve()

PROJECT_TP2_DIR = SCRIPT_DIR.parent
if str(PROJECT_TP2_DIR) not in sys.path:
    sys.path.append(str(PROJECT_TP2_DIR))

from logging_config import LoggerManager

load_dotenv()

USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
STG_DATABASE = os.getenv("STG_DATABASE")
DWH_DATABASE = os.getenv("DWH_DATABASE")

engine_stg = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{STG_DATABASE}",
    poolclass=NullPool,
)

engine_dwh = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DWH_DATABASE}",
    poolclass=NullPool,
)

logger = LoggerManager.configurar("transformacion", ruta_raiz=str(SCRIPT_DIR), carpeta_logs="logs")

TABLAS_DWH = [
    "dim_tiempo",
    "dim_dictado",
    "dim_estudiante",
    "fact_inscripcion",
    "fact_examen_estudiante",
    "fact_evaluacion_dictado",
]

ORDEN_TRUNCATE = [
    "fact_evaluacion_dictado",
    "fact_examen_estudiante",
    "fact_inscripcion",
    "dim_estudiante",
    "dim_dictado",
    "dim_tiempo",
]

MESES_ES = {1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio", 7: "Julio", 8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"}

class DataCleaner:
    @staticmethod
    def limpiar_string(valor) -> Optional[str]:
        if pd.isna(valor) or valor is None: return None
        texto = str(valor).strip()
        if texto.lower() in {"", "null", "none", "n/a", "na", "sin dato", "s/d"}: return None
        return " ".join(texto.split())

    @staticmethod
    def limpiar_numero(valor, tipo: str = "float", requerido: bool = False):
        if pd.isna(valor) or valor is None: return None
        texto = str(valor).strip()
        if texto.lower() in {"", "null", "none", "n/a", "na", "sin dato", "s/d"}: return None
        try:
            texto = texto.replace(".", "") if texto.count(".") > 1 else texto
            texto = texto.replace(",", ".").replace(" ", "")
            numero = float(texto)
            if tipo == "int": return int(numero)
            if tipo == "float": return float(numero)
            return None
        except Exception:
            return None

    @staticmethod
    def limpiar_fecha(valor) -> Optional[date]:
        if pd.isna(valor) or valor is None: return None
        texto = str(valor).strip()
        if texto.lower() in {"", "null", "none", "n/a", "na", "sin dato", "s/d"}: return None
        for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%Y%m%d", "%d-%m-%Y", "%m-%d-%Y", "%Y", "%d/%m/%y", "%Y/%m/%d"]:
            try: return pd.to_datetime(texto, format=fmt).date()
            except Exception: continue
        try: return pd.to_datetime(texto, dayfirst=True).date()
        except Exception: return None

    @staticmethod
    def limpiar_genero(valor) -> Optional[str]:
        texto = DataCleaner.limpiar_string(valor)
        if texto is None: return None
        texto = texto.upper()
        if texto in {"M", "MASCULINO", "MALE", "HOMBRE", "1"}: return "M"
        if texto in {"F", "FEMENINO", "FEMALE", "MUJER", "2"}: return "F"
        if texto in {"X", "OTRO", "OTRA", "NO BINARIO", "NB"}: return "X"
        return None

    @staticmethod
    def limpiar_dni_varchar(valor) -> Optional[str]:
        if pd.isna(valor) or valor is None: return None
        texto = str(valor).strip()
        if texto.lower() in {"", "null", "none", "n/a", "na", "sin dato", "s/d"}: return None
        texto = texto.replace(" ", "")
        return texto if texto else None

    @staticmethod
    def limpiar_nacionalidad(valor) -> str:
        if pd.isna(valor) or valor is None: return "sin especificar"
        texto = str(valor).strip()
        if texto.lower() in {"", "null", "none", "n/a", "na", "sin dato", "s/d"}: return "sin especificar"
        return " ".join(texto.split())

    @staticmethod
    def limpiar_nombre_duplicado(valor) -> Optional[str]:
        texto = DataCleaner.limpiar_string(valor)
        if not texto: return None
        palabras = texto.split()
        if not palabras: return None
        resultado = [palabras[0]]
        for p in palabras[1:]:
            if p.lower() != resultado[-1].lower():
                resultado.append(p)
        return " ".join(resultado)

    @staticmethod
    def validar_dni(dni) -> bool:
        if pd.isna(dni) or dni is None: return False
        try:
            dni_limpio = str(dni).replace(".", "").replace(" ", "")
            return 1_000_000 <= int(dni_limpio) <= 99_999_999
        except Exception: return False

    @staticmethod
    def validar_nota(nota, minimo: float = 0, maximo: float = 10) -> bool:
        if pd.isna(nota) or nota is None: return False
        try: return minimo <= float(nota) <= maximo
        except Exception: return False

    @staticmethod
    def normalizar_estado_inscripcion(valor) -> Optional[str]:
        texto = DataCleaner.limpiar_string(valor)
        if texto is None: return None
        normalizado = texto.strip().lower()
        if normalizado in {"activa", "activo", "inscripto", "inscripta", "cursando", "regular"}: return "Activa"
        if normalizado in {"aprobada", "aprobado", "finalizada", "finalizado"}: return "Aprobada"
        if normalizado in {"abandonada", "abandonado", "abandono", "baja"}: return "Abandonada"
        if normalizado in {"rechazada", "rechazado", "cancelada", "cancelado", "anulada", "anulado"}: return "Cancelada"
        return texto.title()

    @staticmethod
    def normalizar_periodo(valor) -> Optional[int]:
        texto = DataCleaner.limpiar_string(valor)
        if texto is None: return None
        normalizado = texto.strip().upper()
        if normalizado in {"C1", "1", "P1", "PRIMER", "PRIMERO", "PRIMER CUATRIMESTRE"}: return 1
        if normalizado in {"C2", "2", "P2", "SEGUNDO", "SEGUNDO CUATRIMESTRE"}: return 2
        numero = DataCleaner.limpiar_numero(texto, "int")
        if numero in {1, 2}: return int(numero)
        return None

    @staticmethod
    def normalizar_resultado_examen(valor, nota=None) -> Optional[bool]:
        if nota is not None and not pd.isna(nota): return float(nota) >= 6
        texto = DataCleaner.limpiar_string(valor)
        if texto is not None:
            normalizado = texto.lower()
            if normalizado in {"aprobado", "aprob", "sí", "si", "1", "true", "a"}: return True
            if normalizado in {"desaprobado", "desaprob", "no", "0", "false", "d"}: return False
            if normalizado in {"ausente", "pendiente", "no rindio", "no rindió"}: return False
        return None

def estadisticas(total: int, validos: int, duplicados: int = 0, motivo: Optional[str] = None) -> Dict[str, object]:
    return {"total": int(total), "válidos": int(validos), "rechazados": max(int(total) - int(validos) - int(duplicados), 0), "duplicados": int(duplicados)}

def leer_tabla_staging(nombre_tabla: str) -> pd.DataFrame:
    return pd.read_sql(f"SELECT * FROM {nombre_tabla}", con=engine_stg)

def contar_tabla_dwh(nombre_tabla: str) -> int:
    with engine_dwh.connect() as conn:
        return int(conn.execute(text(f"SELECT COUNT(*) FROM {nombre_tabla}")).scalar())

def fecha_desde_anio(anio) -> Optional[date]:
    anio_limpio = DataCleaner.limpiar_numero(anio, "int")
    if anio_limpio is None: return None
    if 1900 <= anio_limpio <= date.today().year + 10: return date(int(anio_limpio), 1, 1)
    return None

def calcular_edad(fecha_nacimiento: Optional[date], anio_ingreso: Optional[int]) -> Optional[int]:
    if not fecha_nacimiento or anio_ingreso is None:
        return None

    try:
        anio_ingreso = int(anio_ingreso)

        # Como no existe fecha exacta de ingreso,
        # se usa el inicio del año académico
        edad = anio_ingreso - fecha_nacimiento.year

        return edad if 0 <= edad <= 120 else None

    except Exception:
        return None

def tiempo_skey(fecha: Optional[date]) -> Optional[int]:
    if not fecha: return None
    return fecha.year * 10000 + fecha.month * 100 + fecha.day

def periodo_academico(fecha: date) -> str:
    return f"{'C1' if fecha.month <= 7 else 'C2'}-{fecha.year}"

def quitar_duplicados(df: pd.DataFrame, subset: List[str], keep: str = "first") -> Tuple[pd.DataFrame, int]:
    antes = len(df)
    limpio = df.drop_duplicates(subset=subset, keep=keep).copy()
    return limpio, antes - len(limpio)

def registrar_rechazos(nombre: str, total: int, validos: int) -> None:
    pass

2026-05-11 20:34:15 - INFO - Iniciando transformacion. Log: E:\Documentos\Facu\2026\ADE\ADE2026_TpiUniversidad\TP2\2-ETL_CargaInicial\logs\transformacion_20260511_203415.log


In [2]:
def transformar_estudiante_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_estudiante"] = df["id_estudiante_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["dni"] = df["dni_raw"].apply(cleaner.limpiar_dni_varchar)
    df["apellido"] = df["apellido_raw"].apply(cleaner.limpiar_nombre_duplicado)
    df["nombre"] = df["nombre_raw"].apply(cleaner.limpiar_nombre_duplicado)
    df["genero"] = df["genero_raw"].apply(cleaner.limpiar_genero)
    df["fecha_nacimiento"] = df["fecha_nacimiento_raw"].apply(cleaner.limpiar_fecha)
    df["nacionalidad"] = df["nacionalidad_raw"].apply(cleaner.limpiar_nacionalidad)
    df["id_programa"] = df["id_programa_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["anio_ingreso"] = df["anio_ingreso_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))

    valido = df["id_estudiante"].notna() & df["dni"].apply(cleaner.validar_dni) & df["apellido"].notna() & df["nombre"].notna() & df["id_programa"].notna()
    validos = df[valido].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_estudiante"], keep="first")
    validos, duplicados_dni = quitar_duplicados(validos, ["dni"], keep="first")
    return validos[["id_estudiante", "dni", "apellido", "nombre", "genero", "fecha_nacimiento", "nacionalidad", "id_programa", "anio_ingreso"]], estadisticas(total, len(validos), duplicados + duplicados_dni)

def transformar_programa_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_programa"] = df["id_programa_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["nombre_programa"] = df["nombre_raw"].apply(cleaner.limpiar_string)
    df["tipo_programa"] = df["tipo_raw"].apply(cleaner.limpiar_string)
    df["duracion_anios_programa"] = df["duracion_anios_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["id_facultad"] = df["id_facultad_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))

    valido = df["id_programa"].notna() & df["nombre_programa"].notna()
    validos = df[valido].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_programa"], keep="first")
    return validos[["id_programa", "nombre_programa", "tipo_programa", "duracion_anios_programa", "id_facultad"]], estadisticas(total, len(validos), duplicados)

def transformar_facultad_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_facultad"] = df["id_facultad_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["nombre_facultad"] = df["nombre_raw"].apply(cleaner.limpiar_string)
    df["ciudad_facultad"] = df["ciudad_raw"].apply(cleaner.limpiar_string)
    df["provincia_facultad"] = df["provincia_raw"].apply(cleaner.limpiar_string)
    validos = df[df["id_facultad"].notna() & df["nombre_facultad"].notna()].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_facultad"], keep="first")
    return validos[["id_facultad", "nombre_facultad", "ciudad_facultad", "provincia_facultad"]], estadisticas(total, len(validos), duplicados)

def transformar_departamento_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_departamento"] = df["id_departamento_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["nombre_departamento"] = df["nombre_raw"].apply(cleaner.limpiar_string)
    df["id_facultad"] = df["id_facultad_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    validos = df[df["id_departamento"].notna() & df["nombre_departamento"].notna()].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_departamento"], keep="first")
    return validos[["id_departamento", "nombre_departamento", "id_facultad"]], estadisticas(total, len(validos), duplicados)

def transformar_docente_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_docente"] = df["id_docente_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["apellido_docente"] = df["apellido_raw"].apply(cleaner.limpiar_string)
    df["nombre_docente"] = df["nombre_raw"].apply(cleaner.limpiar_string)
    df["titulo_docente"] = df["titulo_raw"].apply(cleaner.limpiar_string)
    df["categoria_docente"] = df["categoria_raw"].apply(cleaner.limpiar_string)
    df["dedicacion_docente"] = df["dedicacion_raw"].apply(cleaner.limpiar_string)
    df["id_departamento"] = df["id_departamento_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    validos = df[df["id_docente"].notna() & df["apellido_docente"].notna() & df["nombre_docente"].notna()].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_docente"], keep="first")
    return validos[["id_docente", "apellido_docente", "nombre_docente", "titulo_docente", "categoria_docente", "dedicacion_docente", "id_departamento"]], estadisticas(total, len(validos), duplicados)

def transformar_curso_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_curso"] = df["id_curso_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["codigo_curso"] = df["codigo_raw"].apply(cleaner.limpiar_string)
    df["nombre_curso"] = df["nombre_raw"].apply(cleaner.limpiar_string)
    df["horas_teo_curso"] = df["horas_teorica_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["horas_prac_curso"] = df["horas_ejercicios_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["horas_lab_curso"] = df["horas_laboratorio_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["nivel_curso"] = df["nivel_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    for col in ["horas_teo_curso", "horas_prac_curso", "horas_lab_curso", "nivel_curso"]:
        df.loc[df[col].notna() & (df[col] < 0), col] = None
    validos = df[df["id_curso"].notna() & df["nombre_curso"].notna()].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_curso"], keep="first")
    return validos[["id_curso", "codigo_curso", "nombre_curso", "horas_teo_curso", "horas_prac_curso", "horas_lab_curso", "nivel_curso"]], estadisticas(total, len(validos), duplicados)

def transformar_dictado_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_dictado"] = df["id_dictado_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["id_curso"] = df["id_curso_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["id_docente"] = df["id_docente_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["id_programa"] = df["id_programa_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["periodo"] = df["periodo_raw"].apply(cleaner.normalizar_periodo)
    df["turno"] = df["turno_raw"].apply(cleaner.limpiar_string)
    df["aula"] = df["aula_raw"].apply(cleaner.limpiar_string)
    df["cupo_maximo"] = df["cupo_maximo_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df.loc[df["cupo_maximo"].notna() & (df["cupo_maximo"] < 0), "cupo_maximo"] = None
    validos = df[df["id_dictado"].notna() & df["id_curso"].notna() & df["id_docente"].notna()].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_dictado"], keep="first")
    return validos[["id_dictado", "id_curso", "id_docente", "id_programa", "periodo", "turno", "aula", "cupo_maximo"]], estadisticas(total, len(validos), duplicados)

def transformar_inscripcion_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_inscripcion"] = df["id_inscripcion_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["id_estudiante"] = df["id_estudiante_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["id_dictado"] = df["id_dictado_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["fecha_inscripcion"] = df["fecha_inscripcion_raw"].apply(cleaner.limpiar_fecha)
    df["estado"] = df["estado_raw"].apply(cleaner.normalizar_estado_inscripcion)
    validos = df[df["id_inscripcion"].notna() & df["id_estudiante"].notna() & df["id_dictado"].notna() & df["fecha_inscripcion"].notna()].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_inscripcion"], keep="first")
    return validos[["id_inscripcion", "id_estudiante", "id_dictado", "fecha_inscripcion", "estado"]], estadisticas(total, len(validos), duplicados)

def transformar_examen_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    df["id_examen"] = df["id_examen_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["id_inscripcion"] = df["id_inscripcion_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["fecha"] = df["fecha_raw"].apply(cleaner.limpiar_fecha)
    df["nota"] = df["nota_raw"].apply(lambda x: cleaner.limpiar_numero(x, "float"))
    df["numero_intento"] = df["numero_intento_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["aprobado"] = df.apply(lambda row: cleaner.normalizar_resultado_examen(row["resultado_raw"], row["nota"]), axis=1)
    valido = df["id_examen"].notna() & df["id_inscripcion"].notna() & df["fecha"].notna() & df["nota"].apply(lambda x: cleaner.validar_nota(x, 0, 10)) & df["numero_intento"].notna() & (df["numero_intento"] > 0)
    validos = df[valido].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_examen"], keep="first")
    return validos[["id_examen", "id_inscripcion", "fecha", "nota", "numero_intento", "aprobado"]], estadisticas(total, len(validos), duplicados)

def transformar_evaluacion_base(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    cleaner = DataCleaner()
    total = len(df)
    df = df.copy()
    for col in ["id_estudiante_raw", "fecha_evaluacion_raw"]:
        if col not in df.columns: df[col] = None
    df["id_evaluacion"] = df["id_evaluacion_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["id_dictado"] = df["id_dictado_raw"].apply(lambda x: cleaner.limpiar_numero(x, "int"))
    df["fecha_evaluacion"] = df["fecha_evaluacion_raw"].apply(cleaner.limpiar_fecha)
    df["puntaje_dictado"] = df["puntaje_dictado_raw"].apply(lambda x: cleaner.limpiar_numero(x, "float"))
    df["puntaje_contenido"] = df["puntaje_contenido_raw"].apply(lambda x: cleaner.limpiar_numero(x, "float"))
    df["valoracion_general"] = df["valoracion_general_raw"].apply(lambda x: cleaner.limpiar_numero(x, "float"))
    puntajes_validos = df["puntaje_dictado"].apply(lambda x: cleaner.validar_nota(x, 0, 10)) & df["puntaje_contenido"].apply(lambda x: cleaner.validar_nota(x, 0, 10)) & df["valoracion_general"].apply(lambda x: cleaner.validar_nota(x, 0, 10))
    valido = df["id_evaluacion"].notna() & df["id_dictado"].notna() & df["fecha_evaluacion"].notna() & puntajes_validos
    validos = df[valido].copy()
    validos, duplicados = quitar_duplicados(validos, ["id_evaluacion"], keep="first")
    return validos[["id_evaluacion", "id_dictado", "fecha_evaluacion", "puntaje_dictado", "puntaje_contenido", "valoracion_general"]], estadisticas(total, len(validos), duplicados)

In [3]:
def construir_dim_tiempo(fechas: Iterable[Optional[date]]) -> Tuple[pd.DataFrame, Dict]:
    fechas_validas = sorted({f for f in fechas if f is not None and not pd.isna(f)})
    registros = [{"tiempo_skey": tiempo_skey(f), "fecha": f, "dia": f.day, "mes": MESES_ES[f.month], "anio": f.year, "periodo_academico": periodo_academico(f), "es_feriado": False} for f in fechas_validas]
    df_tiempo = pd.DataFrame(registros)
    return df_tiempo, estadisticas(len(fechas_validas), len(df_tiempo), 0)

def construir_dim_estudiante(estudiantes: pd.DataFrame, programas: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    total = len(estudiantes)
    df = estudiantes.merge(programas, on="id_programa", how="left")
    df["edadIngreso"] = df.apply(lambda row: calcular_edad(row["fecha_nacimiento"], row["anio_ingreso"]), axis=1)
    df["valid_from"] = date.today()
    df["valid_to"] = None
    df["es_actual"] = True
    resultado = pd.DataFrame({"id_estudiante": df["id_estudiante"], "dni": df["dni"], "nombre": df["nombre"], "apellido": df["apellido"], "genero": df["genero"], "fecha_nac": df["fecha_nacimiento"], "nacionalidad": df["nacionalidad"], "anio_ingreso": df["anio_ingreso"], "edad_ingreso": df["edadIngreso"], "egreso_carrera": False, "anio_egreso": None, "abandono_carrera": False, "anio_abandono": None, "nombre_prog": df["nombre_programa"], "tipo_prog": df["tipo_programa"], "duracion_prog": df["duracion_anios_programa"], "anio_plan_prog": None, "valid_from": df["valid_from"], "valid_to": df["valid_to"], "es_actual": df["es_actual"]})
    resultado, duplicados = quitar_duplicados(resultado, ["id_estudiante"], keep="first")
    return resultado, estadisticas(total, len(resultado), duplicados)

def construir_dim_dictado(dictados: pd.DataFrame, cursos: pd.DataFrame, docentes: pd.DataFrame, departamentos: pd.DataFrame, facultades: pd.DataFrame) -> Tuple[pd.DataFrame, Dict]:
    total = len(dictados)
    df = dictados.merge(cursos, on="id_curso", how="left").merge(docentes, on="id_docente", how="left").merge(departamentos, on="id_departamento", how="left").merge(facultades, on="id_facultad", how="left")
    resultado = pd.DataFrame({"id_dictado": df["id_dictado"], "periodo": df["periodo"], "turno": df["turno"], "aula": df["aula"], "cupo_max": df["cupo_maximo"], "codigo_curso": df["codigo_curso"], "nombre_curso": df["nombre_curso"], "horas_teoria": df["horas_teo_curso"], "horas_practica": df["horas_prac_curso"], "horas_lab": df["horas_lab_curso"], "nivel_curso": df["nivel_curso"], "nombre_docente": df["nombre_docente"], "apellido_docente": df["apellido_docente"], "titulo_docente": df["titulo_docente"], "categoria_docente": df["categoria_docente"], "dedicacion_docente": df["dedicacion_docente"], "nombre_dpto": df["nombre_departamento"], "nombre_fac": df["nombre_facultad"], "ciudad_fac": df["ciudad_facultad"], "prov_fac": df["provincia_facultad"], "valid_from": date.today(), "valid_to": None, "es_actual": True})
    resultado, duplicados = quitar_duplicados(resultado, ["id_dictado"], keep="first")
    return resultado, estadisticas(total, len(resultado), duplicados)

def obtener_mapa_estudiante() -> Dict[int, int]:
    df = pd.read_sql("SELECT estudiante_skey, id_estudiante FROM dim_estudiante WHERE es_actual = TRUE", con=engine_dwh)
    return dict(zip(df["id_estudiante"], df["estudiante_skey"]))

def obtener_mapa_dictado() -> Dict[int, int]:
    df = pd.read_sql("SELECT dictado_skey, id_dictado FROM dim_dictado WHERE es_actual = TRUE", con=engine_dwh)
    return dict(zip(df["id_dictado"], df["dictado_skey"]))

def obtener_mapa_tiempo() -> Dict[date, int]:
    df = pd.read_sql("SELECT tiempo_skey, fecha FROM dim_tiempo", con=engine_dwh)
    df["fecha"] = pd.to_datetime(df["fecha"]).dt.date
    return dict(zip(df["fecha"], df["tiempo_skey"]))

In [ ]:
def construir_fact_inscripcion(inscripciones: pd.DataFrame, mapa_estudiante: Dict[int, int], mapa_dictado: Dict[int, int], mapa_tiempo: Dict[date, int]) -> Tuple[pd.DataFrame, Dict]:
    total = len(inscripciones)
    df = inscripciones.copy()
    df["estudiante_skey"] = df["id_estudiante"].map(mapa_estudiante)
    df["dictado_skey"] = df["id_dictado"].map(mapa_dictado)
    df["tiempo_skey"] = df["fecha_inscripcion"].map(mapa_tiempo)
    df["abandono"] = df["estado"].fillna("").str.lower().isin(["abandonada", "abandonado", "abandono", "baja"])
    valido = df["estudiante_skey"].notna() & df["dictado_skey"].notna() & df["tiempo_skey"].notna()
    validos = df[valido].copy()
    resultado = validos[["estudiante_skey", "tiempo_skey", "dictado_skey", "estado", "abandono"]].copy()
    resultado, duplicados = quitar_duplicados(resultado, ["estudiante_skey", "dictado_skey"], keep="last")
    return resultado, estadisticas(total, len(resultado), duplicados)

def construir_fact_examen_estudiante(examenes: pd.DataFrame, inscripciones: pd.DataFrame, mapa_estudiante: Dict[int, int], mapa_dictado: Dict[int, int], mapa_tiempo: Dict[date, int]) -> Tuple[pd.DataFrame, Dict]:
    total = len(examenes)
    df = examenes.merge(inscripciones[["id_inscripcion", "id_estudiante", "id_dictado"]], on="id_inscripcion", how="left")
    df["estudiante_skey"] = df["id_estudiante"].map(mapa_estudiante)
    df["dictado_skey"] = df["id_dictado"].map(mapa_dictado)
    df["tiempo_skey"] = df["fecha"].map(mapa_tiempo)
    valido = df["estudiante_skey"].notna() & df["dictado_skey"].notna() & df["tiempo_skey"].notna()
    validos = df[valido].copy()
    resultado = validos[["estudiante_skey", "tiempo_skey", "dictado_skey", "nota", "numero_intento", "aprobado"]].copy()
    resultado = resultado.rename(columns={"numero_intento": "n_intentos"})
    # Regla de negocio:
    # una inscripción no puede tener dos veces el mismo intento
    resultado, duplicados = quitar_duplicados(
        resultado,
        ["estudiante_skey", "dictado_skey", "n_intentos"],
        keep="last"
    )
    return resultado, estadisticas(total, len(resultado), duplicados)

def construir_fact_evaluacion_dictado(evaluaciones: pd.DataFrame, mapa_dictado: Dict[int, int], mapa_tiempo: Dict[date, int]) -> Tuple[pd.DataFrame, Dict]:
    total = len(evaluaciones)
    df = evaluaciones.copy()
    df["dictado_skey"] = df["id_dictado"].map(mapa_dictado)
    df["tiempo_skey"] = df["fecha_evaluacion"].map(mapa_tiempo)
    valido = df["dictado_skey"].notna() & df["tiempo_skey"].notna()
    validos = df[valido].copy()
    validos = validos.sort_values(["fecha_evaluacion", "id_evaluacion"])
    resultado = validos[["dictado_skey", "tiempo_skey", "puntaje_dictado", "puntaje_contenido", "valoracion_general"]].copy()
    resultado = resultado.rename(columns={"puntaje_dictado": "nota_dictado", "puntaje_contenido": "nota_cont", "valoracion_general": "nota_general"})
    duplicados = 0
    return resultado, estadisticas(total, len(resultado), duplicados)

def truncar_dwh() -> None:
    with engine_dwh.begin() as conn:
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
        for tabla in ORDEN_TRUNCATE:
            conn.execute(text(f"TRUNCATE TABLE {tabla}"))
        conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))

def insertar_dataframe(df: pd.DataFrame, tabla_destino: str, tamanio_lote: int = 500) -> Dict:
    if df.empty: return {"insertados": 0, "errores": 0, "lotes": 0}
    df = df.where(pd.notnull(df), None)
    resultados = {"insertados": 0, "errores": 0, "lotes": 0}
    num_lotes = (len(df) + tamanio_lote - 1) // tamanio_lote
    for i in range(num_lotes):
        lote = df.iloc[i * tamanio_lote:min((i + 1) * tamanio_lote, len(df))].copy()
        try:
            lote.to_sql(name=tabla_destino, con=engine_dwh, if_exists="append", index=False, method="multi")
            resultados["insertados"] += len(lote)
            resultados["lotes"] += 1
        except Exception as e:
            resultados["errores"] += len(lote)

            logger.exception(
                f"""
                Error insertando lote en {tabla_destino}

                Lote: {i + 1}/{num_lotes}
                Registros: {len(lote)}

                Error:
                {str(e)}
                """
            )

            for _, fila in lote.iterrows():
                try:
                    pd.DataFrame([fila]).to_sql(
                        name=tabla_destino,
                        con=engine_dwh,
                        if_exists="append",
                        index=False
                    )
                    resultados["insertados"] += 1

                except Exception as fila_error:
                    resultados["errores"] += 1
                    logger.exception(
                        f"""
                        Error insertando fila individual en {tabla_destino}

                        Datos:
                        {fila.to_dict()}

                        Error:
                        {str(fila_error)}
                        """
                    )

    return resultados

def cargar_tabla(nombre_tabla: str, df: pd.DataFrame, reporte: Dict, stats_transformacion: Dict) -> None:
    count = contar_tabla_dwh(nombre_tabla)
    if count > 0:
        with engine_dwh.begin() as conn:
            conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
            conn.execute(text(f"TRUNCATE TABLE {nombre_tabla}"))
            conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))
    stats_insert = insertar_dataframe(df, nombre_tabla)
    reporte[nombre_tabla] = {"transformacion": stats_transformacion, "insercion": stats_insert, "final": contar_tabla_dwh(nombre_tabla)}

def ejecutar_transformacion() -> Dict:
    reporte: Dict = {}
    staging_transformaciones = {
        "facultades": ("stg_facultad", transformar_facultad_base),
        "departamentos": ("stg_departamento", transformar_departamento_base),
        "programas": ("stg_programa", transformar_programa_base),
        "cursos": ("stg_curso", transformar_curso_base),
        "docentes": ("stg_docente", transformar_docente_base),
        "estudiantes": ("stg_estudiante", transformar_estudiante_base),
        "dictados": ("stg_dictado", transformar_dictado_base),
        "inscripciones": ("stg_inscripcion", transformar_inscripcion_base),
        "examenes": ("stg_examen", transformar_examen_base),
        "evaluaciones": ("stg_evaluacion_curso", transformar_evaluacion_base),
    }

    datos: Dict[str, pd.DataFrame] = {}
    stats_base: Dict[str, Dict] = {}

    for clave, (tabla_stg, funcion) in staging_transformaciones.items():
        df_raw = leer_tabla_staging(tabla_stg)
        df_limpio, stats = funcion(df_raw)
        datos[clave] = df_limpio
        stats_base[tabla_stg] = stats

    reporte["staging_limpieza"] = stats_base

    fechas_tiempo = []
    if not datos["inscripciones"].empty: fechas_tiempo.extend(datos["inscripciones"]["fecha_inscripcion"].tolist())
    if not datos["examenes"].empty: fechas_tiempo.extend(datos["examenes"]["fecha"].tolist())
    if not datos["evaluaciones"].empty: fechas_tiempo.extend(datos["evaluaciones"]["fecha_evaluacion"].tolist())

    dim_tiempo, stats_tiempo = construir_dim_tiempo(fechas_tiempo)
    dim_estudiante, stats_estudiante = construir_dim_estudiante(datos["estudiantes"], datos["programas"])
    dim_dictado, stats_dictado = construir_dim_dictado(datos["dictados"], datos["cursos"], datos["docentes"], datos["departamentos"], datos["facultades"])

    truncar_dwh()

    cargar_tabla("dim_tiempo", dim_tiempo, reporte, stats_tiempo)
    cargar_tabla("dim_estudiante", dim_estudiante, reporte, stats_estudiante)
    cargar_tabla("dim_dictado", dim_dictado, reporte, stats_dictado)

    mapa_estudiante = obtener_mapa_estudiante()
    mapa_dictado = obtener_mapa_dictado()
    mapa_tiempo = obtener_mapa_tiempo()

    fact_inscripcion, stats_fact_inscripcion = construir_fact_inscripcion(datos["inscripciones"], mapa_estudiante, mapa_dictado, mapa_tiempo)
    fact_examen, stats_fact_examen = construir_fact_examen_estudiante(datos["examenes"], datos["inscripciones"], mapa_estudiante, mapa_dictado, mapa_tiempo)
    fact_evaluacion, stats_fact_evaluacion = construir_fact_evaluacion_dictado(datos["evaluaciones"], mapa_dictado, mapa_tiempo)

    cargar_tabla("fact_inscripcion", fact_inscripcion, reporte, stats_fact_inscripcion)
    cargar_tabla("fact_examen_estudiante", fact_examen, reporte, stats_fact_examen)
    cargar_tabla("fact_evaluacion_dictado", fact_evaluacion, reporte, stats_fact_evaluacion)

    imprimir_reporte(reporte)
    return reporte

def imprimir_reporte(reporte: Dict) -> None:
    print("=" * 80)
    print("REPORTE GENERAL DE CARGA DWH")
    print("=" * 80)
    total_leidos = 0
    total_validos = 0
    total_insertados = 0
    total_errores = 0

    for tabla, stats in reporte.items():
        if tabla == "staging_limpieza": continue
        print(f"\n Tabla: {tabla}")
        if 'transformacion' in stats:
            total_validos += stats['transformacion']['válidos']
            print(f"  Transformacion: válidos={stats['transformacion']['válidos']}, duplicados={stats['transformacion']['duplicados']}")
        if 'insercion' in stats:
            total_insertados += stats['insercion']['insertados']
            total_errores += stats['insercion']['errores']
            print(f"  Insercion: insertados={stats['insercion']['insertados']}, errores={stats['insercion']['errores']}")
        print(f"  Final en BD: {stats['final']}")

    print("\n RESUMEN GLOBAL:")
    print(f"Total registros válidos a cargar: {total_validos}")
    print(f"Total registros insertados exitosamente: {total_insertados}")
    print(f"Total errores durante inserción: {total_errores}")
    if total_errores == 0:
        print("[OK] TRANSFORMACION EXITOSA")
    else:
        print("[WARN] HUBO ERRORES DE INSERCION")

# Ejecutar el flujo principal
if __name__ == "__main__":

    try:

        logger.info("Iniciando ETL")

        ejecutar_transformacion()

        logger.info("ETL finalizado correctamente")

    except Exception as e:

        logger.exception(f"Error general ETL: {e}")

        raise

2026-05-11 20:34:15 - INFO - Iniciando ETL
